# UdaPlay — Part 1: RAG Pipeline

Build and verify the knowledge base for the UdaPlay gaming research agent.

## What this notebook does

1. **Load** 25 game records from `games.json`
2. **Embed** each record with OpenAI `text-embedding-ada-002` (via ChromaDB)
3. **Store** in a ChromaDB in-memory collection
4. **Query** to verify semantic search works end-to-end

The resulting `VectorStore` object is reused in `Udaplay_02_solution_project.ipynb`.

## Framework

This project follows the patterns from the *Building Agents* course:
- `lib.vector_db.VectorStoreManager` — manages ChromaDB collections
- `lib.vector_db.CorpusLoaderService` — loads datasets into vector stores
- `lib.loaders.JSONGameLoader` — converts JSON game records to `Document` objects
- `lib.state_machine.StateMachine` — powers the RAG retrieve→augment→generate loop
- `lib.rag.RAG` — the complete RAG pipeline as a reusable component

In [1]:
# Uncomment if dependencies are not yet installed
# !pip install chromadb>=1.0.4 openai>=1.73.0 pydantic>=2.11.3 python-dotenv>=1.1.0 tavily-python>=0.5.4 pdfplumber

In [2]:
# Only needed for Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
from dotenv import load_dotenv

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY must be set in config.env"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY must be set in config.env"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

print("✓ Environment loaded")
print(f"  Base URL: {OPENAI_BASE_URL}")

✓ Environment loaded
  Base URL: https://openai.vocareum.com/v1


In [4]:
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.llm import LLM
from lib.state_machine import Run

print("✓ lib imports successful")

✓ lib imports successful


## Step 1 — Initialize the Vector Store Manager

In [5]:
# VectorStoreManager handles ChromaDB client setup and OpenAI embedding function.
# Pass api_base so the embedding calls route through the Vocareum proxy.

db = VectorStoreManager(OPENAI_API_KEY, api_base=OPENAI_BASE_URL)
print(db)

VectorStoreManager():<chromadb.api.client.Client object at 0x7bf666ee52b0>


In [6]:
loader_service = CorpusLoaderService(db)
print("✓ CorpusLoaderService ready")

✓ CorpusLoaderService ready


## Step 2 — Load and embed the games dataset

`CorpusLoaderService.load_json()` uses `JSONGameLoader` under the hood:
1. Reads every game record from `games.json`
2. Converts each record into a natural-language `Document`
3. Batches all documents into a ChromaDB collection (OpenAI embeddings are generated automatically)

In [7]:
games_store = loader_service.load_json(
    store_name="games",
    json_path="games.json",
)

VectorStore `games` ready!


210 games from `games.json` added!


## Step 3 — Verify semantic search

Query the vector store directly and inspect similarity scores.

In [8]:
import json

def print_search_results(query: str, n_results: int = 3) -> None:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    raw = games_store.query(query_texts=[query], n_results=n_results)
    docs  = raw["documents"][0] if raw["documents"] else []
    dists = raw["distances"][0] if raw["distances"] else []
    metas = raw["metadatas"][0] if raw["metadatas"] else []

    for i, (doc, dist, meta) in enumerate(zip(docs, dists, metas), 1):
        title = meta.get("title", "Unknown")
        dev   = meta.get("developer", "Unknown")
        year  = meta.get("release_date", "?")[:4]
        print(f"  {i}. [similarity {1-dist:.4f}] {title} — {dev} ({year})")


queries = [
    "Who developed FIFA 21?",
    "What platform was Pokemon Red originally launched on?",
    "When was God of War Ragnarok released?",
    "Open-world action RPG games",
    "Games developed by Rockstar Games",
    "First-person shooter with campaign mode",
    "Nintendo exclusive life simulation games",
]

for q in queries:
    print_search_results(q)


Query: 'Who developed FIFA 21?'
------------------------------------------------------------


  1. [similarity 0.7859] FIFA 21 — EA Sports (2020)
  2. [similarity 0.5708] Final Fantasy XVI — Square Enix Creative Business Unit III (2023)
  3. [similarity 0.5640] Fortnite — Epic Games (2017)

Query: 'What platform was Pokemon Red originally launched on?'
------------------------------------------------------------


  1. [similarity 0.7441] Pokemon Red — Game Freak (1996)
  2. [similarity 0.6560] Pokemon Gold and Silver — Game Freak (1999)
  3. [similarity 0.6556] Pokemon Scarlet and Violet — Game Freak (2022)

Query: 'When was God of War Ragnarok released?'
------------------------------------------------------------


  1. [similarity 0.7843] God of War Ragnarok — Santa Monica Studio (2022)
  2. [similarity 0.7471] God of War (2018) — Santa Monica Studio (2018)
  3. [similarity 0.7145] God of War III — Santa Monica Studio (2010)

Query: 'Open-world action RPG games'
------------------------------------------------------------


  1. [similarity 0.6795] Cyberpunk 2077 — CD Projekt Red (2020)
  2. [similarity 0.6719] World of Warcraft — Blizzard Entertainment (2004)
  3. [similarity 0.6681] The Elder Scrolls V: Skyrim — Bethesda Game Studios (2011)

Query: 'Games developed by Rockstar Games'
------------------------------------------------------------


  1. [similarity 0.7384] Grand Theft Auto IV — Rockstar North (2008)
  2. [similarity 0.7330] Grand Theft Auto V — Rockstar North (2013)
  3. [similarity 0.7163] Grand Theft Auto III — DMA Design (Rockstar North) (2001)

Query: 'First-person shooter with campaign mode'
------------------------------------------------------------


  1. [similarity 0.6816] Call of Duty: Modern Warfare (2019) — Infinity Ward (2019)
  2. [similarity 0.6766] Call of Duty: Modern Warfare 3 — Infinity Ward / Sledgehammer Games (2011)
  3. [similarity 0.6695] Call of Duty: Modern Warfare II — Infinity Ward (2022)

Query: 'Nintendo exclusive life simulation games'
------------------------------------------------------------


  1. [similarity 0.6751] Animal Crossing: New Horizons — Nintendo EPD (2020)
  2. [similarity 0.6411] Wii Sports — Nintendo EAD (2006)
  3. [similarity 0.6404] Pikmin 4 — Nintendo EPD (2023)


## Step 4 — RAG pipeline demo

Wrap the vector store in a `RAG` pipeline (retrieve → augment → generate)
to produce natural-language answers backed by the retrieved context.

In [9]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

games_rag = RAG(
    llm=rag_llm,
    vector_store=games_store,
)

print("✓ RAG pipeline ready")

✓ RAG pipeline ready


In [10]:
# RAG query 1 — developer lookup
result: Run = games_rag.invoke("Who developed FIFA 21?")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
FIFA 21 was developed by EA Sports.


In [11]:
# RAG query 2 — platform and release date
result: Run = games_rag.invoke(
    "What platform was Pokémon Red originally launched on and in what year?"
)
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Pokémon Red was originally launched on the Game Boy in the year 1996.


In [12]:
# RAG query 3 — genre-based discovery
result: Run = games_rag.invoke(
    "List the open-world RPG games in the dataset with their release dates."
)
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
The open-world RPG games in the dataset with their release dates are:

1. The Elder Scrolls V: Skyrim - Release Date: 2011-11-11
2. World of Warcraft - Release Date: 2004-11-23
3. Dragon Age: Origins - Release Date: 2009-11-03

(Note: While World of Warcraft is primarily an MMORPG, it has elements of open-world gameplay.)


## Step 5 — Dataset statistics

In [ ]:
raw_all = games_store.get(limit=300)
all_metas = raw_all["metadatas"] or []
all_games = [json.loads(m["json_data"]) for m in all_metas]

print(f"Total games indexed: {len(all_games)}")

genre_counts: dict = {}
year_counts: dict = {}
for g in all_games:
    genre_counts[g.get("genre", "Unknown")] = genre_counts.get(g.get("genre", "Unknown"), 0) + 1
    year = g.get("release_date", "?")[:4]
    year_counts[year] = year_counts.get(year, 0) + 1

print("\nGenre breakdown:")
for genre, count in sorted(genre_counts.items(), key=lambda x: -x[1]):
    print(f"  {genre:<40} {'█' * count} ({count})")

print("\nRelease year distribution:")
for year in sorted(year_counts):
    print(f"  {year}  {'█' * year_counts[year]} ({year_counts[year]})")

print("\n✓ RAG pipeline verified — run Udaplay_02_solution_project.ipynb for the agent.")